# 🎬 CineMatch — Sistema híbrido de recomendación de películas

## 👋 ¿Qué hace este notebook?

Este notebook sirve para **probar, entender y validar** cómo funciona **CineMatch**, un sistema de recomendación de películas **híbrido**.

El recomendador combina varias ideas para dar mejores resultados:

- 🎭 **Contenido**: usa géneros, rating medio y popularidad
- 🤝 **Colaborativo**: aprovecha patrones aprendidos con MovieLens
- 🧠 **Usuarios similares**: busca comportamientos parecidos
- 🧊 **Cold start**: recomienda incluso si el usuario no tiene ratings
- 🔄 **Rotación controlada**: si repites la misma petición, no devuelve siempre exactamente lo mismo

---

## ⚙️ ¿Cómo funciona el flujo?

El sistema sigue estos pasos:

1. 📥 Recibe los datos del usuario  
   - gustos por género  
   - ratings dados por el usuario  
   - número de recomendaciones deseadas  

2. 🎯 Genera una **shortlist inicial** con contenido

3. 🤖 Aplica señales colaborativas sobre esa shortlist  
   - score SVD / embeddings  
   - score por usuarios similares  

4. ⚖️ Mezcla todas las señales en un **score final**

5. 🔄 Aplica una pequeña rotación para evitar tops repetitivos

6. 📤 Devuelve las recomendaciones finales

---

## 🧩 Ficheros del proyecto

Este notebook trabaja con estos módulos:

- `collaborative_svd.py` → modelo colaborativo con embeddings
- `content_based.py` → lógica basada en contenido
- `user_based_embeddings.py` → usuarios similares
- `hybrid_recommender.py` → orquestador central del sistema
- `demo_modelo_hibrido.py` → utilidades de prueba y simulación HTTP

---

## 📁 Datos esperados

Para que todo funcione, normalmente necesitas:

- `movies_df` con:
  - `movieId`
  - `title`
  - `genres`
  - `rating`
  - `num_ratings`

- `ratings_df` con:
  - `userId`
  - `movieId`
  - `rating`

- un **checkpoint del modelo entrenado**

- opcionalmente `user_ratings_df` con los ratings del usuario actual

---

## 🧪 ¿Qué se prueba aquí?

En este notebook podrás probar:

- 🆕 usuario nuevo sin ratings
- ⭐ usuario con gustos por género
- 🎞️ usuario con ratings propios
- 🔁 cómo cambian las recomendaciones si repites la misma petición
- 📊 métricas, comparaciones y simulación tipo HTTP

---

## 🎯 Objetivo

La idea de este notebook es ver, de forma clara, cómo CineMatch:

- recibe datos del usuario
- decide qué señales usar
- mezcla contenido + colaborativo
- evita repetir siempre las mismas películas
- devuelve recomendaciones listas para usar en una API

---

## 🌐 Ejemplo de petición HTTP esperada

El endpoint `/recommend` debe recibir un JSON con esta forma:

```json
{
  "user_id": 999002,
  "user_preferences": {
    "Sci-Fi": 5,
    "Thriller": 4,
    "Drama": 2
  },
  "ratings": [
    { "movieId": 296, "rating": 5.0 },
    { "movieId": 318, "rating": 4.5 },
    { "movieId": 593, "rating": 4.0 },
    { "movieId": 2571, "rating": 5.0 }
  ],
  "top_n": 10,
  "shortlist_size": 300,
  "recommendation_state": null
}
```
---

## 🌐 Ejemplo de petición HTTP devuelta

El endpoint `/recommend` devuelve  un JSON con esta forma:

```json

{
  "recommendations": [
    {
      "movieId": 1196,
      "title": "Star Wars: Episode V - The Empire Strikes Back (1980)",
      "genres": "Action|Adventure|Sci-Fi",
      "rating": 4.2,
      "num_ratings": 129000,
      "baseline_score": 0.81,
      "svd_score": 0.76,
      "embedding_score": 0.73,
      "final_score": 0.78,
      "rotated_score": 0.77,
      "rotation_penalty": 0.02,
      "rotation_jitter": 0.004
    },
    {
      "movieId": 260,
      "title": "Star Wars: Episode IV - A New Hope (1977)",
      "genres": "Action|Adventure|Sci-Fi",
      "rating": 4.1,
      "num_ratings": 125000,
      "baseline_score": 0.79,
      "svd_score": 0.75,
      "embedding_score": 0.71,
      "final_score": 0.77,
      "rotated_score": 0.75,
      "rotation_penalty": 0.03,
      "rotation_jitter": -0.002
    }
  ],
  "metadata": {
    "num_user_ratings": 4,
    "has_preferences": true,
    "used_cold_start": false,
    "request_signature": "abc123"
  },
  "recommendation_state": {
    "signatures": {
      "abc123": {
        "request_count": 1,
        "exposures": {
          "1196": 1.0,
          "260": 1.0
        }
      }
    }
  }
}
```

## 1) Importaciones

In [10]:

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Ajusta estas importaciones si cambias la ubicación del notebook.
import os

# sube 1 nivel desde el archivo actual
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))


ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from collaborative_svd import Recommender
from content_based import enrich_movies_with_stats, compute_baseline_scores
from demo_modelo_hibrido import build_payload, simulate_local_request, ratings_to_dataframe
from hybrid_recommender import HybridRecommender
from user_based_embeddings import UserBasedEmbeddingsRecommender


## 2) Configuración de rutas

In [11]:

def pick_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None

# ------------------------------------------------------------
# Puedes fijar estas rutas manualmente si quieres.
# ------------------------------------------------------------
MODEL_PATH = None
MOVIES_PATH = None
RATINGS_PATH = None

# ------------------------------------------------------------
# Búsqueda automática en rutas comunes del proyecto.
# ------------------------------------------------------------
model_candidates = [
    ROOT / "svd_collaborative_model_v1.pth",
    ROOT / "../svd_collaborative_model_v1.pth",
    ROOT / "src" / "models" / "svd_collaborative_model_v1.pth",
    ROOT.parent / "src" / "models" / "svd_collaborative_model_v1.pth",
    ROOT.parent.parent / "src" / "models" / "svd_collaborative_model_v1.pth",
]

movies_candidates = [
    ROOT / "movies_clean.csv",
    ROOT / "data" / "processed" / "movies_clean.csv",
    ROOT.parent / "data" / "processed" / "movies_clean.csv",
    ROOT.parent.parent / "data" / "processed" / "movies_clean.csv",
    ROOT.parent.parent.parent / "data" / "processed" / "movies_clean.csv",
]

ratings_candidates = [
    ROOT / "ratings_clean.csv",
    ROOT / "data" / "processed" / "ratings_clean.csv",
    ROOT.parent / "data" / "processed" / "ratings_clean.csv",
    ROOT.parent.parent / "data" / "processed" / "ratings_clean.csv",
    ROOT.parent.parent.parent / "data" / "processed" / "ratings_clean.csv",
]

MODEL_PATH = MODEL_PATH or pick_existing_path(model_candidates)
MOVIES_PATH = MOVIES_PATH or pick_existing_path(movies_candidates)
RATINGS_PATH = RATINGS_PATH or pick_existing_path(ratings_candidates)

print("MODEL_PATH  =", MODEL_PATH)
print("MOVIES_PATH =", MOVIES_PATH)
print("RATINGS_PATH=", RATINGS_PATH)

if MODEL_PATH is None or MOVIES_PATH is None or RATINGS_PATH is None:
    raise ValueError(
        "No se han encontrado automáticamente todas las rutas. "
        "Edita esta celda y fija MODEL_PATH / MOVIES_PATH / RATINGS_PATH."
    )


MODEL_PATH  = /home/zanwel/proyectos/notebooks/CineMatch/CineMatch-Sistema-de-recomendacion-peliculas/src/models/notebooks/../svd_collaborative_model_v1.pth
MOVIES_PATH = /home/zanwel/proyectos/notebooks/CineMatch/CineMatch-Sistema-de-recomendacion-peliculas/data/processed/movies_clean.csv
RATINGS_PATH= /home/zanwel/proyectos/notebooks/CineMatch/CineMatch-Sistema-de-recomendacion-peliculas/data/processed/ratings_clean.csv


## 3) Carga y preparación de datos

In [12]:

movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)

movies_df = enrich_movies_with_stats(movies_df, ratings_df)

print("Movies shape :", movies_df.shape)
print("Ratings shape:", ratings_df.shape)

display(movies_df.head(3))
display(ratings_df.head(3))


Movies shape : (87585, 5)
Ratings shape: (31895825, 4)


,movieId,title,genres,rating,num_ratings
0,1,Toy Story (1995),"['Adventure', 'Animation', 'Children', 'Comedy...",3.897436,68996
1,2,Jumanji (1995),"['Adventure', 'Children', 'Fantasy']",3.275767,28903
2,3,Grumpier Old Men (1995),"['Comedy', 'Romance']",3.139447,13134


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


## 4) Inicialización del sistema

In [13]:

hybrid = HybridRecommender(
    model_path=str(MODEL_PATH),
    ratings_df=ratings_df,
    movies_df=movies_df,
)

svd_rec = hybrid.rec
embedding_cf = hybrid.emb_cf

print("Modelo cargado correctamente.")
print("Usuarios en entrenamiento:", len(svd_rec.user_to_index))
print("Películas en entrenamiento:", len(svd_rec.movie_to_index))
print("Media global del modelo:", round(svd_rec.global_mean, 4))


Modelo cargado correctamente.
Usuarios en entrenamiento: 200947
Películas en entrenamiento: 43883
Media global del modelo: 3.5423


## 5) Helpers de evaluación y depuración

In [14]:

def show_path_info(result):
    metadata = result["metadata"]
    print("=" * 80)
    print("CAMINO ELEGIDO :", metadata.get("path"))
    print("PESOS          :", metadata.get("weights"))
    print("RATINGS USER   :", metadata.get("num_user_ratings"))
    print("PREFERENCIAS   :", metadata.get("num_preferences"))
    print("ROTACIÓN       :", metadata.get("rotation"))
    print("USER TRAINED?  :", metadata.get("trained_user_available"))
    print("TEMP PROFILE?  :", metadata.get("used_temp_profile"))
    print("=" * 80)

def overlap_at_k(df_a, df_b, k=10):
    a = set(df_a["movieId"].head(k).tolist()) if not df_a.empty else set()
    b = set(df_b["movieId"].head(k).tolist()) if not df_b.empty else set()
    return len(a & b)

def distinct_genres_in_top(df, k=10):
    if df.empty:
        return 0
    genres = set()
    for value in df["genres"].head(k).tolist():
        if isinstance(value, str):
            for g in value.replace("|", ",").replace("[", "").replace("]", "").replace("'", "").split(","):
                g = g.strip()
                if g:
                    genres.add(g.lower())
    return len(genres)

def summarize_recommendations(df, k=10):
    if df.empty:
        return pd.DataFrame([{
            "top_k": k,
            "avg_rating_catalog": 0.0,
            "avg_num_ratings": 0.0,
            "distinct_genres": 0,
        }])

    top = df.head(k).copy()
    return pd.DataFrame([{
        "top_k": k,
        "avg_rating_catalog": round(float(top["rating"].astype(float).mean()), 4),
        "avg_num_ratings": round(float(top["num_ratings"].astype(float).mean()), 2),
        "distinct_genres": distinct_genres_in_top(top, k=k),
    }])

def build_user_ratings(movie_ids, ratings):
    return pd.DataFrame({
        "movieId": movie_ids,
        "rating": ratings,
    })

def compare_models_table(content_df, svd_df, emb_df, hybrid_df, k=10):
    frames = []
    for name, df in [
        ("content", content_df),
        ("svd_temp", svd_df),
        ("neighbors", emb_df),
        ("hybrid", hybrid_df),
    ]:
        summary = summarize_recommendations(df, k=k)
        summary.insert(0, "model", name)
        frames.append(summary)

    return pd.concat(frames, ignore_index=True)


## 6) Caso A — Cold start puro

In [15]:

cold_result = hybrid.recommend_with_metadata(
    user_id=999999,
    user_preferences={"Action": 5, "Adventure": 4, "Sci-Fi": 5},
    user_ratings_df=pd.DataFrame(columns=["movieId", "rating"]),
    top_n=10,
)

show_path_info(cold_result)
display(cold_result["recommendations_df"].head(10))
display(summarize_recommendations(cold_result["recommendations_df"]))


CAMINO ELEGIDO : cold_start_content_only
PESOS          : {'svd': 0.0, 'embedding': 0.0, 'content': 1.0}
RATINGS USER   : 0
PREFERENCIAS   : 3
ROTACIÓN       : {'signature': 'b56eeb4a4cf5a2c39c8ad87b3d1daa55d37a499add12a6a405a7bc4f55663c0e', 'request_count': 1, 'tracked_movies': 10}
USER TRAINED?  : False
TEMP PROFILE?  : False


,movieId,baseline_score,title,genres,rating,num_ratings,genre_score,rating_score,popularity_score,svd_score,embedding_score,final_score,rotated_score,rotation_penalty,rotation_jitter
0,1196,0.686200,Star Wars: Episode V - The Empire Strikes Back...,"['Action', 'Adventure', 'Sci-Fi']",4.130353,72150,0.390208,0.909648,0.961146,0.0,0.0,0.686200,0.700474,0.0,0.014274
1,260,0.686296,Star Wars: Episode IV - A New Hope (1977),"['Action', 'Adventure', 'Sci-Fi']",4.099825,85009,0.390208,0.899674,0.979082,0.0,0.0,0.686296,0.696350,0.0,0.010054
2,1210,0.668758,Star Wars: Episode VI - Return of the Jedi (1983),"['Action', 'Adventure', 'Sci-Fi']",3.991555,67495,0.390208,0.863982,0.953852,0.0,0.0,0.668758,0.661953,0.0,-0.006804
3,1198,0.658990,Raiders of the Lost Ark (Indiana Jones and the...,"['Action', 'Adventure']",4.111895,67407,0.337792,0.903548,0.953709,0.0,0.0,0.658990,0.647931,0.0,-0.011059
4,1291,0.636371,Indiana Jones and the Last Crusade (1989),"['Action', 'Adventure']",3.986864,46400,0.337792,0.862258,0.912869,0.0,0.0,0.636371,0.644480,0.0,0.008110
5,112852,0.638734,Guardians of the Galaxy (2014),"['Action', 'Adventure', 'Sci-Fi']",3.910181,26481,0.390208,0.836667,0.851532,0.0,0.0,0.638734,0.642736,0.0,0.004002
6,2571,0.640754,"Matrix, The (1999)","['Action', 'Sci-Fi', 'Thriller']",4.156438,93807,0.269707,0.918330,0.989852,0.0,0.0,0.640754,0.640368,0.0,-0.000385
7,589,0.642624,Terminator 2: Judgment Day (1991),"['Action', 'Sci-Fi']",3.962044,68382,0.339040,0.854284,0.955280,0.0,0.0,0.642624,0.639917,0.0,-0.002707
8,59315,0.634898,Iron Man (2008),"['Action', 'Adventure', 'Sci-Fi']",3.815148,36467,0.390208,0.805713,0.886525,0.0,0.0,0.634898,0.638412,0.0,0.003514
9,1200,0.634976,Aliens (1986),"['Action', 'Adventure', 'Horror', 'Sci-Fi']",4.007260,38845,0.338208,0.868844,0.893433,0.0,0.0,0.634976,0.624505,0.0,-0.010471


,top_k,avg_rating_catalog,avg_num_ratings,distinct_genres
0,10,4.0172,60244.3,5


## 7) Caso B — Usuario externo con pocos ratings

In [16]:

few_ratings_df = build_user_ratings(
    movie_ids=[1, 32, 47, 50, 296],
    ratings=[4.5, 4.0, 5.0, 4.5, 5.0],
)

few_result = hybrid.recommend_with_metadata(
    user_id=123456789,  # usuario externo, no tiene por qué existir en MovieLens
    user_preferences={"Action": 4, "Thriller": 5, "Drama": 4, "Crime": 5},
    user_ratings_df=few_ratings_df,
    top_n=10,
)

show_path_info(few_result)
display(few_result["recommendations_df"].head(10))
display(summarize_recommendations(few_result["recommendations_df"]))


CAMINO ELEGIDO : hybrid_content_plus_temp_svd_plus_neighbors
PESOS          : {'svd': 0.35, 'embedding': 0.25, 'content': 0.4}
RATINGS USER   : 5
PREFERENCIAS   : 4
ROTACIÓN       : {'signature': '0d64d6431c66dc165b43755e0f2cddc09ca6ecea1596f8d1d990742dc03b1d94', 'request_count': 1, 'tracked_movies': 10}
USER TRAINED?  : False
TEMP PROFILE?  : True


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331,0.698870,1.000000,1.000000,0.879548,0.0,-0.009935,0.869613
1,318,"Shawshank Redemption, The (1994)","['Crime', 'Drama']",4.404613,102928,0.694206,0.992486,0.969814,0.867506,0.0,-0.010279,0.857227
2,593,"Silence of the Lambs, The (1991)","['Crime', 'Horror', 'Thriller']",4.148369,90329,0.631194,0.989922,0.958718,0.838630,0.0,0.011481,0.850111
3,2571,"Matrix, The (1999)","['Action', 'Sci-Fi', 'Thriller']",4.156438,93807,0.632392,0.981976,0.946421,0.833254,0.0,0.011106,0.844360
4,858,"Godfather, The (1972)","['Crime', 'Drama']",4.317028,66439,0.674477,0.992652,0.894387,0.840816,0.0,0.000760,0.841576
5,1221,"Godfather: Part II, The (1974)","['Crime', 'Drama']",4.264463,43110,0.658865,0.974597,0.826433,0.811263,0.0,0.008404,0.819668
6,2329,American History X (1998),"['Crime', 'Drama']",4.130447,38966,0.641218,0.933465,0.888867,0.805417,0.0,0.011112,0.816529
7,1213,Goodfellas (1990),"['Crime', 'Drama']",4.188420,42002,0.649545,0.959589,0.871467,0.813541,0.0,-0.001544,0.811997
8,58559,"Dark Knight, The (2008)","['Action', 'Crime', 'Drama', 'IMAX']",4.173344,59333,0.651361,0.958861,0.883147,0.816932,0.0,-0.007124,0.809809
9,48516,"Departed, The (2006)","['Crime', 'Drama', 'Thriller']",4.131375,35235,0.658781,0.919761,0.853051,0.798691,0.0,0.009337,0.808028


,top_k,avg_rating_catalog,avg_num_ratings,distinct_genres
0,10,4.2143,64948.0,7


## 8) Caso C — Usuario externo con más ratings

In [17]:

more_ratings_df = build_user_ratings(
    movie_ids=[1, 32, 47, 50, 110, 296, 318, 457, 593, 608, 858, 912, 1196, 1198, 1210],
    ratings=[4.5, 4.0, 5.0, 4.5, 3.5, 5.0, 5.0, 4.0, 4.5, 4.0, 5.0, 4.5, 5.0, 5.0, 4.5],
)

more_result = hybrid.recommend_with_metadata(
    user_id=555000111,
    user_preferences={"Action": 5, "Sci-Fi": 4, "Drama": 4, "Thriller": 4},
    user_ratings_df=more_ratings_df,
    top_n=10,
)

show_path_info(more_result)
display(more_result["recommendations_df"].head(10))
display(summarize_recommendations(more_result["recommendations_df"]))


CAMINO ELEGIDO : hybrid_content_plus_temp_svd_plus_neighbors
PESOS          : {'svd': 0.45, 'embedding': 0.3, 'content': 0.25}
RATINGS USER   : 15
PREFERENCIAS   : 4
ROTACIÓN       : {'signature': '8b3f83e03419b68d3aa1def97641986b0f8598f495753fc26806b9064065f20c', 'request_count': 1, 'tracked_movies': 10}
USER TRAINED?  : False
TEMP PROFILE?  : True


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,260,Star Wars: Episode IV - A New Hope (1977),"['Action', 'Adventure', 'Sci-Fi']",4.099825,85009,0.624374,1.000000,1.000000,0.906093,0.0,-0.004020,0.902073
1,527,Schindler's List (1993),"['Drama', 'War']",4.236987,73848,0.597162,0.981067,0.946798,0.874810,0.0,0.010891,0.885701
2,2571,"Matrix, The (1999)","['Action', 'Sci-Fi', 'Thriller']",4.156438,93807,0.683578,0.976783,0.900297,0.880536,0.0,0.004893,0.885429
3,1221,"Godfather: Part II, The (1974)","['Crime', 'Drama']",4.264463,43110,0.588398,0.975119,0.954040,0.872115,0.0,-0.003635,0.868480
4,2028,Saving Private Ryan (1998),"['Action', 'Drama', 'War']",4.048735,58367,0.610189,0.969818,0.912948,0.862850,0.0,-0.001088,0.861762
5,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331,0.664246,0.958508,0.889867,0.864350,0.0,-0.009370,0.854980
6,1193,One Flew Over the Cuckoo's Nest (1975),['Drama'],4.204223,44591,0.608321,0.932697,0.830853,0.821050,0.0,0.013863,0.834913
7,589,Terminator 2: Judgment Day (1991),"['Action', 'Sci-Fi']",3.962044,68382,0.634908,0.956221,0.808746,0.831650,0.0,0.001178,0.832828
8,1213,Goodfellas (1990),"['Crime', 'Drama']",4.188420,42002,0.579073,0.943926,0.836862,0.820593,0.0,0.010655,0.831249
9,2858,American Beauty (1999),"['Drama', 'Romance']",4.093052,65157,0.577826,0.934854,0.836940,0.816223,0.0,0.014829,0.831051


,top_k,avg_rating_catalog,avg_num_ratings,distinct_genres
0,10,4.1483,65160.4,8


## 9) Comparación de señales por separado

In [18]:

prefs = {"Action": 5, "Sci-Fi": 4, "Drama": 4, "Thriller": 4}
seen_ids = set(more_ratings_df["movieId"].tolist())

candidate_ids = [
    int(movie_id)
    for movie_id in movies_df["movieId"].astype(int).tolist()
    if int(movie_id) not in seen_ids
]

content_df = compute_baseline_scores(
    movie_ids=candidate_ids,
    movies_df=movies_df,
    user_preferences=prefs,
    min_votes=10,
).head(10)

svd_df = svd_rec.recommend(
    user_id=555000111,
    seen_movie_ids=seen_ids,
    movies_df=movies_df,
    top_n=10,
    user_ratings_df=more_ratings_df,
)

emb_df = embedding_cf.recommend(
    user_id=555000111,
    seen_movie_ids=seen_ids,
    top_n=10,
    user_ratings_df=more_ratings_df,
)

hybrid_df = more_result["recommendations_df"]

display(compare_models_table(content_df, svd_df, emb_df, hybrid_df, k=10))

print("Overlap contenido vs híbrido:", overlap_at_k(content_df, hybrid_df, k=10))
print("Overlap svd vs híbrido      :", overlap_at_k(svd_df, hybrid_df, k=10))
print("Overlap vecinos vs híbrido  :", overlap_at_k(emb_df, hybrid_df, k=10))


,model,top_k,avg_rating_catalog,avg_num_ratings,distinct_genres
0,content,10,4.0652,58798.6,8
1,svd_temp,10,4.1347,73979.8,10
2,neighbors,10,4.1574,68351.7,9
3,hybrid,10,4.1483,65160.4,8


Overlap contenido vs híbrido: 4
Overlap svd vs híbrido      : 6
Overlap vecinos vs híbrido  : 9


## 10) Demostración de rotación / no repetición exacta

In [19]:

rotation_state = None
rotation_runs = []

for run_id in range(1, 4):
    result = hybrid.recommend_with_metadata(
        user_id=777000,
        user_preferences={"Action": 5, "Adventure": 4, "Sci-Fi": 5, "Drama": 3},
        user_ratings_df=more_ratings_df,
        top_n=10,
        recommendation_state=rotation_state,  # importante para APIs stateless
        apply_rotation=True,
    )
    rotation_state = result["recommendation_state"]
    df_run = result["recommendations_df"].copy()
    df_run["run"] = run_id
    rotation_runs.append(df_run)

rotation_df = pd.concat(rotation_runs, ignore_index=True)

for run_id in sorted(rotation_df["run"].unique()):
    print(f"\n--- RUN {run_id} ---")
    display(rotation_df[rotation_df["run"] == run_id][[
        "movieId", "title", "final_score", "rotated_score",
        "rotation_penalty", "rotation_jitter"
    ]].head(10))

run1 = rotation_df[rotation_df["run"] == 1]
run2 = rotation_df[rotation_df["run"] == 2]
run3 = rotation_df[rotation_df["run"] == 3]

print("Overlap run1 vs run2:", overlap_at_k(run1, run2, k=10))
print("Overlap run2 vs run3:", overlap_at_k(run2, run3, k=10))
print("Overlap run1 vs run3:", overlap_at_k(run1, run3, k=10))



--- RUN 1 ---


,movieId,title,final_score,rotated_score,rotation_penalty,rotation_jitter
0,260,Star Wars: Episode IV - A New Hope (1977),0.918815,0.918915,0.0,0.000100
1,1221,"Godfather: Part II, The (1974)",0.871714,0.876895,0.0,0.005181
2,2571,"Matrix, The (1999)",0.868220,0.875027,0.0,0.006807
3,527,Schindler's List (1993),0.874365,0.870637,0.0,-0.003728
4,2028,Saving Private Ryan (1998),0.862933,0.855550,0.0,-0.007383
5,2959,Fight Club (1999),0.853839,0.849187,0.0,-0.004652
6,356,Forrest Gump (1994),0.814847,0.823183,0.0,0.008336
7,589,Terminator 2: Judgment Day (1991),0.832115,0.819715,0.0,-0.012400
8,1193,One Flew Over the Cuckoo's Nest (1975),0.821057,0.817421,0.0,-0.003637
9,1213,Goodfellas (1990),0.820424,0.815327,0.0,-0.005097



--- RUN 2 ---


,movieId,title,final_score,rotated_score,rotation_penalty,rotation_jitter
10,260,Star Wars: Episode IV - A New Hope (1977),0.918815,0.863551,0.068,0.012736
11,1291,Indiana Jones and the Last Crusade (1989),0.811925,0.823460,0.000,0.011535
12,527,Schindler's List (1993),0.874365,0.812554,0.068,0.006189
13,2571,"Matrix, The (1999)",0.868220,0.811974,0.068,0.011753
14,58559,"Dark Knight, The (2008)",0.815541,0.811070,0.000,-0.004471
15,2858,American Beauty (1999),0.816120,0.804724,0.000,-0.011397
16,1200,Aliens (1986),0.799578,0.801245,0.000,0.001667
17,1221,"Godfather: Part II, The (1974)",0.871714,0.799541,0.068,-0.004173
18,2028,Saving Private Ryan (1998),0.862933,0.797142,0.068,0.002209
19,1197,"Princess Bride, The (1987)",0.783679,0.795708,0.000,0.012028



--- RUN 3 ---


,movieId,title,final_score,rotated_score,rotation_penalty,rotation_jitter
20,2959,Fight Club (1999),0.853839,0.810142,0.0544,0.010703
21,7153,"Lord of the Rings: The Return of the King, The...",0.792575,0.794768,0.0000,0.002193
22,1214,Alien (1979),0.783730,0.787016,0.0000,0.003286
23,1240,"Terminator, The (1984)",0.774545,0.785002,0.0000,0.010457
24,260,Star Wars: Episode IV - A New Hope (1977),0.918815,0.783955,0.1224,-0.012460
25,1193,One Flew Over the Cuckoo's Nest (1975),0.821057,0.776815,0.0544,0.010158
26,589,Terminator 2: Judgment Day (1991),0.832115,0.776622,0.0544,-0.001093
27,356,Forrest Gump (1994),0.814847,0.773620,0.0544,0.013173
28,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.777232,0.771188,0.0000,-0.006044
29,541,Blade Runner (1982),0.777806,0.770151,0.0000,-0.007656


Overlap run1 vs run2: 5
Overlap run2 vs run3: 1
Overlap run1 vs run3: 5


## 11) Pruebas pequeñas de robustez

In [20]:

# 1) usuario sin preferencias pero con ratings
result_no_prefs = hybrid.recommend_with_metadata(
    user_id=8080,
    user_preferences={},
    user_ratings_df=more_ratings_df.head(8),
    top_n=5,
)
show_path_info(result_no_prefs)
display(result_no_prefs["recommendations_df"])

# 2) top_n inválido
empty_case = hybrid.recommend_with_metadata(
    user_id=1,
    user_preferences={},
    user_ratings_df=more_ratings_df,
    top_n=0,
)
print("Caso top_n=0:", empty_case["metadata"])

# 3) ratings de películas que no estén en el modelo (si no existen, simplemente se ignoran)
weird_ratings_df = pd.DataFrame({
    "movieId": [999999999, 888888888, 1, 50],
    "rating": [5.0, 4.0, 4.5, 5.0],
})
weird_result = hybrid.recommend_with_metadata(
    user_id=9090,
    user_preferences={"Drama": 5, "Crime": 4},
    user_ratings_df=weird_ratings_df,
    top_n=5,
)
show_path_info(weird_result)
display(weird_result["recommendations_df"])


CAMINO ELEGIDO : hybrid_content_plus_temp_svd_plus_neighbors
PESOS          : {'svd': 0.48, 'embedding': 0.27, 'content': 0.25}
RATINGS USER   : 8
PREFERENCIAS   : 0
ROTACIÓN       : {'signature': '3036da7a53160745af74ef16431aae33272fd3536435df860819823ee77476d8', 'request_count': 1, 'tracked_movies': 5}
USER TRAINED?  : True
TEMP PROFILE?  : True


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,593,"Silence of the Lambs, The (1991)","['Crime', 'Horror', 'Thriller']",4.148369,90329,0.941366,0.999280,1.000000,0.984996,0.0,0.004118,0.989114
1,2571,"Matrix, The (1999)","['Action', 'Sci-Fi', 'Thriller']",4.156438,93807,0.944551,0.987934,0.964913,0.970873,0.0,0.013337,0.984210
2,356,Forrest Gump (1994),"['Comedy', 'Drama', 'Romance', 'War']",4.052749,100295,0.924954,0.997953,0.975149,0.973547,0.0,0.009046,0.982592
3,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331,0.952559,1.000000,0.989177,0.985218,0.0,-0.006418,0.978800
4,527,Schindler's List (1993),"['Drama', 'War']",4.236987,73848,0.952529,0.994594,0.989305,0.982650,0.0,-0.013085,0.969564


Caso top_n=0: {'path': 'empty_top_n', 'weights': {'svd': 0.0, 'embedding': 0.0, 'content': 0.0}}
CAMINO ELEGIDO : hybrid_content_plus_temp_svd_plus_neighbors
PESOS          : {'svd': 0.25, 'embedding': 0.2, 'content': 0.55}
RATINGS USER   : 4
PREFERENCIAS   : 2
ROTACIÓN       : {'signature': 'b7c83ac6a2c1a0693d5501b01f549de89867fc76a714d738fb265e9441bcf0c6', 'request_count': 1, 'tracked_movies': 5}
USER TRAINED?  : True
TEMP PROFILE?  : True


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,318,"Shawshank Redemption, The (1994)","['Crime', 'Drama']",4.404613,102928,0.717606,1.000000,1.000000,0.844684,0.0,0.003088,0.847771
1,858,"Godfather, The (1972)","['Crime', 'Drama']",4.317028,66439,0.697877,0.984498,0.862282,0.802413,0.0,0.002214,0.804627
2,296,Pulp Fiction (1994),"['Comedy', 'Crime', 'Drama', 'Thriller']",4.196966,98408,0.645911,0.974325,0.965357,0.791904,0.0,0.011165,0.803069
3,527,Schindler's List (1993),"['Drama', 'War']",4.236987,73848,0.610091,0.993276,0.952138,0.774297,0.0,-0.002987,0.771310
4,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331,0.644269,0.928252,0.823995,0.751210,0.0,0.012649,0.763859


## 12) Simulación local de petición tipo HTTP

In [21]:

payload = build_payload(
    user_id=424242,
    user_preferences={
        "Action": 5,
        "Drama": 4,
        "Thriller": 5,
        "Crime": 4,
    },
    ratings=[
        {"movieId": 1, "rating": 4.5},
        {"movieId": 32, "rating": 4.0},
        {"movieId": 47, "rating": 5.0},
        {"movieId": 50, "rating": 5.0},
        {"movieId": 296, "rating": 4.5},
        {"movieId": 318, "rating": 5.0},
    ],
    top_n=8,
    shortlist_size=250,
)

local_response = simulate_local_request(hybrid, payload)


display(local_response["dataframe"].head(8))

print("JSON devuelto por la simulación local:")
print(json.dumps(local_response["json"]["metadata"], indent=2, ensure_ascii=False))


PAYLOAD LOCAL
{
  "user_id": 424242,
  "user_preferences": {
    "Action": 5,
    "Drama": 4,
    "Thriller": 5,
    "Crime": 4
  },
  "ratings": [
    {
      "movieId": 1,
      "rating": 4.5
    },
    {
      "movieId": 32,
      "rating": 4.0
    },
    {
      "movieId": 47,
      "rating": 5.0
    },
    {
      "movieId": 50,
      "rating": 5.0
    },
    {
      "movieId": 296,
      "rating": 4.5
    },
    {
      "movieId": 318,
      "rating": 5.0
    }
  ],
  "top_n": 8,
  "shortlist_size": 250,
  "recommendation_state": null
}


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331.0,0.699532,1.000000,1.000000,0.879813,0.0,0.007958,0.887771
1,593,"Silence of the Lambs, The (1991)","['Crime', 'Horror', 'Thriller']",4.148369,90329.0,0.631302,0.994179,0.984176,0.846527,0.0,0.010653,0.857180
2,858,"Godfather, The (1972)","['Crime', 'Drama']",4.317028,66439.0,0.673262,0.994404,0.927837,0.849305,0.0,0.004104,0.853409
3,2571,"Matrix, The (1999)","['Action', 'Sci-Fi', 'Thriller']",4.156438,93807.0,0.633626,0.984179,0.950439,0.835523,0.0,0.008597,0.844120
4,527,Schindler's List (1993),"['Drama', 'War']",4.236987,73848.0,0.597161,0.988815,0.961012,0.825203,0.0,0.013358,0.838561
5,1213,Goodfellas (1990),"['Crime', 'Drama']",4.188420,42002.0,0.648296,0.951650,0.962225,0.832952,0.0,-0.007684,0.825268
6,1221,"Godfather: Part II, The (1974)","['Crime', 'Drama']",4.264463,43110.0,0.657621,0.972390,0.880216,0.823439,0.0,-0.001442,0.821996
7,79132,Inception (2010),"['Action', 'Crime', 'Drama', 'Mystery', 'Sci-F...",4.157172,57930.0,0.644787,0.931674,0.949990,0.821498,0.0,-0.001235,0.820263


JSON devuelto por la simulación local:
{
  "path": "hybrid_content_plus_temp_svd_plus_neighbors",
  "weights": {
    "svd": 0.35,
    "embedding": 0.25,
    "content": 0.4
  },
  "num_user_ratings": 6,
  "num_preferences": 4,
  "candidate_pool_size": 87579,
  "shortlist_size": 250,
  "trained_user_available": false,
  "used_temp_profile": true,
  "svd_candidates_scored": 250,
  "embedding_candidates_scored": 131,
  "rotation": {
    "signature": "f2570392908bc3583b8f3a03d22cec5fe6a4bdd99e1d3c0148fd473846762660",
    "request_count": 1,
    "tracked_movies": 8
  }
}


## 13) Segunda llamada idéntica usando el estado devuelto

In [22]:

payload_round_2 = dict(payload)
payload_round_2["recommendation_state"] = local_response["json"]["recommendation_state"]

local_response_round_2 = simulate_local_request(hybrid, payload_round_2, show_request=False)

print("Metadatos llamada 2:")
print(json.dumps(local_response_round_2["json"]["metadata"], indent=2, ensure_ascii=False))

display(local_response_round_2["dataframe"].head(8))

print("Overlap llamada 1 vs llamada 2:",
      overlap_at_k(local_response["dataframe"], local_response_round_2["dataframe"], k=8))


Metadatos llamada 2:
{
  "path": "hybrid_content_plus_temp_svd_plus_neighbors",
  "weights": {
    "svd": 0.35,
    "embedding": 0.25,
    "content": 0.4
  },
  "num_user_ratings": 6,
  "num_preferences": 4,
  "candidate_pool_size": 87579,
  "shortlist_size": 250,
  "trained_user_available": false,
  "used_temp_profile": true,
  "svd_candidates_scored": 250,
  "embedding_candidates_scored": 131,
  "rotation": {
    "signature": "f2570392908bc3583b8f3a03d22cec5fe6a4bdd99e1d3c0148fd473846762660",
    "request_count": 2,
    "tracked_movies": 15
  }
}


,movieId,title,genres,rating,num_ratings,baseline_score,svd_score,embedding_score,final_score,rotation_penalty,rotation_jitter,rotated_score
0,58559,"Dark Knight, The (2008)","['Action', 'Crime', 'Drama', 'IMAX']",4.173344,59333.0,0.652005,0.964014,0.885161,0.819497,0.000,0.000099,0.819596
1,2028,Saving Private Ryan (1998),"['Action', 'Drama', 'War']",4.048735,58367.0,0.610187,0.972598,0.871786,0.802431,0.000,0.010674,0.813105
2,2329,American History X (1998),"['Crime', 'Drama']",4.130447,38966.0,0.639962,0.939984,0.915560,0.813869,0.000,-0.001039,0.812831
3,2959,Fight Club (1999),"['Action', 'Crime', 'Drama', 'Thriller']",4.228789,77331.0,0.699532,1.000000,1.000000,0.879813,0.068,-0.001131,0.810682
4,1089,Reservoir Dogs (1992),"['Crime', 'Mystery', 'Thriller']",4.092673,43729.0,0.608850,0.951894,0.865053,0.792966,0.000,0.013324,0.806290
5,2858,American Beauty (1999),"['Drama', 'Romance']",4.093052,65157.0,0.577824,0.980061,0.872790,0.792349,0.000,0.011358,0.803706
6,356,Forrest Gump (1994),"['Comedy', 'Drama', 'Romance', 'War']",4.052749,100295.0,0.560201,0.988172,0.889312,0.792269,0.000,0.006009,0.798278
7,6016,City of God (Cidade de Deus) (2002),"['Action', 'Adventure', 'Crime', 'Drama', 'Thr...",4.178651,25687.0,0.650563,0.918345,0.783029,0.777403,0.000,0.014718,0.792121


Overlap llamada 1 vs llamada 2: 1



## 14) Conclusiones rápidas

Qué deberías mirar cuando pruebes tu sistema:

- **`metadata["path"]`** para saber qué camino eligió el híbrido.
- **`metadata["weights"]`** para ver cuánto peso tuvo cada señal.
- **`recommendation_state`** si quieres que una API stateless pueda rotar resultados entre llamadas.
- **`rotation_penalty`** y **`rotated_score`** para entender por qué una película baja o sube entre repeticiones.
- **comparativas entre contenido / SVD temporal / vecinos / híbrido** para detectar sesgos.

Si ves mucho sesgo a un género:
- baja peso de contenido,
- sube shortlist,
- revisa `compute_genre_score`,
- o aumenta ligeramente la penalización de rotación.
